In [1]:
import numpy as np
import pandas as pd
import os
os.chdir('../')

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import string

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

import spacy
#nlp = spacy.load("en_core_web_sm")

# Sentiment analysis libraries
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from afinn import Afinn

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.pipeline import Pipeline

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

data = pd.read_csv('Datasets/AirlineTweets.csv')
data = data[['airline_sentiment','text']]
target_map = {'positive':1, 'neutral': 0, 'negative':-1}
data['target'] = data.airline_sentiment.map(target_map)
data = data.drop(['airline_sentiment'],axis=1)
data.head()

,text,target
0,@VirginAmerica What @dhepburn said.,0
1,@VirginAmerica plus you've added commercials t...,1
2,@VirginAmerica I didn't today... Must mean I n...,0
3,@VirginAmerica it's really aggressive to blast...,-1
4,@VirginAmerica and it's a really big bad thing...,-1


In [8]:
from textblob import TextBlob

TextBlob('Today is a good day').sentiment


Sentiment(polarity=0.7, subjectivity=0.6000000000000001)

In [12]:
TextBlob('Life is going well').sentiment

Sentiment(polarity=0.0, subjectivity=0.0)

In [16]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\athar\AppData\Roaming\nltk_data...


True

In [17]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

vader=SentimentIntensityAnalyzer()
vader.polarity_scores('Arnold is a bodybuilder as well as an actor.')


{'neg': 0.0, 'neu': 0.769, 'pos': 0.231, 'compound': 0.2732}

In [18]:
X_train, X_test, y_train, y_test = train_test_split(data.text, 
                                                    data.target,
                                                    test_size = 0.8, 
                                                    random_state=1031)

In [19]:
text = 'My trip was riddled with problems. The flight itself was delayed at takeoff, the seats were uncomfortable, food was stale and ride was bumpy'
from textblob import TextBlob
blob = TextBlob(text)
text, blob.sentiment.polarity, blob.sentiment.subjectivity

('My trip was riddled with problems. The flight itself was delayed at takeoff, the seats were uncomfortable, food was stale and ride was bumpy',
 -0.5,
 0.75)

In [20]:
from textblob import TextBlob
def sentiment_textblob(text):
    blob = TextBlob(text)
    return(blob.sentiment.polarity)

sentiment_train_textblob = X_train.apply(sentiment_textblob)
sentiment_test_textblob = X_test.apply(sentiment_textblob)

sentiment_train_textblob = pd.cut(pd.Series(sentiment_train_textblob), bins = 3, labels = [-1,0,1])
sentiment_test_textblob = pd.cut(pd.Series(sentiment_test_textblob), bins = 3, labels = [-1,0,1])

sentiment_train_textblob[:5], sentiment_test_textblob[:5]

(11262    0
 13233    0
 12742    0
 12691   -1
 1532     0
 Name: text, dtype: category
 Categories (3, int64): [-1 < 0 < 1],
 5274    0
 2543    0
 4908    0
 7547    0
 8654    1
 Name: text, dtype: category
 Categories (3, int64): [-1 < 0 < 1])

In [21]:
from sklearn.metrics import accuracy_score
accuracy_textblob = accuracy_score(y_test, sentiment_test_textblob)
accuracy_textblob

0.3364924863387978

In [22]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()
text, vader.polarity_scores(text)

('My trip was riddled with problems. The flight itself was delayed at takeoff, the seats were uncomfortable, food was stale and ride was bumpy',
 {'neg': 0.255, 'neu': 0.745, 'pos': 0.0, 'compound': -0.7351})

In [23]:
def sentiment_vader(text):
    return SentimentIntensityAnalyzer().polarity_scores(text)['compound']
sentiment_train_vader = X_train.apply(sentiment_vader)
sentiment_test_vader = X_test.apply(sentiment_vader)

sentiment_train_vader = pd.cut(pd.Series(sentiment_train_vader), bins = 3, labels = [-1,0,1])
sentiment_test_vader = pd.cut(pd.Series(sentiment_test_vader), bins = 3, labels = [-1,0,1])

sentiment_train_vader[:5], sentiment_test_vader[:5]

(11262    0
 13233   -1
 12742    1
 12691   -1
 1532     0
 Name: text, dtype: category
 Categories (3, int64): [-1 < 0 < 1],
 5274    0
 2543    1
 4908    0
 7547    0
 8654    0
 Name: text, dtype: category
 Categories (3, int64): [-1 < 0 < 1])

In [24]:
accuracy_vader = accuracy_score(y_test, sentiment_test_vader)
accuracy_vader

0.4718237704918033

In [25]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, accuracy_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data.text, 
                                                    data.target,
                                                    test_size = 0.2, 
                                                    random_state=1031)

tf_vectorizer = CountVectorizer(max_features=2000)
tf_vectorizer.fit(X_train)
X_train_dtm = tf_vectorizer.transform(X_train)
X_test_dtm = tf_vectorizer.transform(X_test)

logit = LogisticRegression()
logit.fit(X_train_dtm, y_train)

pred = logit.predict(X_test_dtm)
#pd.Series(pred).value_counts()
accuracy_score(y_test, pred)

c:\Users\athar\anaconda4\envs\ds_env_311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7923497267759563